<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/07_Customer_Lifetime_Value/01_Silent_Attrition_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 7: Predicting Silent Attrition & Customer Lifetime Value

## The Business Problem: "Silent Attrition"
In subscription-based businesses (like SaaS or streaming), churn is explicit: a customer actively cancels their contract. However, grocery retail is a **non-contractual** domain. Customers do not notify the store when they decide to shop at a competitor; they simply stop showing up. This is known as "Silent Attrition."

Furthermore, measuring this absence requires deep personalization:
* If a **Power Shopper** (who historically visits every 3 days) is absent for 14 days, there is a high probability they have churned.
* If a **Premium Shopper** (who historically visits once a month for specialty items) is absent for 14 days, their behavior is completely normal.

Traditional binary churn models fail because they apply a universal time threshold to all users.

## The Solution: "Buy 'Til You Die" (BTYD)
To solve this, we implement the **BG/NBD (Beta Geometric / Negative Binomial Distribution)** statistical framework. Instead of predicting a binary "Churned vs. Active" state, this probabilistic model calculates the unique "heartbeat" of every individual shopper.

It evaluates customers based on three factors:
1. **Recency:** The age of the customer at their last purchase.
2. **Frequency:** The number of repeat purchases they have made.
3. **Monetary Value:** Their average basket size.

## Project Execution Steps
* **Step 1: RFM Calibration:** Compress 1.4 million transactional scans into a standardized Recency, Frequency, and Monetary (RFM) matrix.
* **Step 2: Probability Modeling:** Train the BG/NBD model to calculate $P(Alive)$—the mathematical probability that a specific customer is still an active shopper.
* **Step 3: Persona Intervention:** Cross-reference high-flight-risk customers with our existing segmentation clusters to identify exactly *which* types of shoppers the business is losing.

## Step 1: RFM Calibration (Recency, Frequency, Monetary)

To feed our data into the BG/NBD probability model, we must transform the raw transactional log into a specialized summary matrix.

Using the `lifetimes` library, we compress the chronological receipt data into four mathematical vectors per customer:
* **Frequency ($x$):** Count of repeat shopping days.
* **Recency ($t_x$):** Time elapsed between the first and last purchase.
* **Age ($T$):** Total time observed from the first purchase to the end of the dataset.
* **Monetary Value ($m$):** Average spend across repeat purchases.

In [6]:
!pip install completejourney_py

In [7]:
# 1. Install the BTYD modeling library
!pip install lifetimes

import pandas as pd
from completejourney_py import get_data
from lifetimes.utils import summary_data_from_transaction_data

# 2. Fetch the raw transactions
print("Fetching transaction data...")
data = get_data()
transactions = data["transactions"]

# 3. Clean the date formatting
transactions['transaction_timestamp'] = pd.to_datetime(transactions['transaction_timestamp'])
transactions['date'] = transactions['transaction_timestamp'].dt.date

# 4. Generate the BTYD Summary Matrix
# We compress the data by household_id, grouping by day.
print("Calibrating RFM Matrix...")
rfm_matrix = summary_data_from_transaction_data(
    transactions,
    customer_id_col='household_id',
    datetime_col='date',
    monetary_value_col='sales_value',
    freq='D' # Daily frequency evaluation
)

print(f"✅ Matrix built! We have calibrated {len(rfm_matrix)} unique households.")
display(rfm_matrix.head(10))

Fetching transaction data...
Calibrating RFM Matrix...
✅ Matrix built! We have calibrated 2469 unique households.


,frequency,recency,T,monetary_value
household_id,,,,
1,46.0,358.0,359.0,50.908478
2,19.0,331.0,359.0,52.668947
3,19.0,349.0,359.0,53.153158
4,17.0,339.0,361.0,24.387647
5,16.0,289.0,349.0,18.023750
6,135.0,361.0,361.0,25.546148
7,31.0,344.0,349.0,62.280000
8,58.0,359.0,362.0,51.942931
9,10.0,349.0,353.0,61.785000


#Explanation:
Looking at the extremes in our output:

Household 1: Made 46 repeat purchases. Their first and last purchase were 358 days apart ($Recency$), and they have been a known customer for 359 days ($Age$). This means they shopped yesterday. They are highly active.

Household 10: Made 0 repeat purchases. They bought once, 152 days ago, and never came back. In retail, this is known as a "One-and-Done" shopper.

Now that we have the math, we can bring in the BG/NBD Model. This algorithm learns the dropout rate of the entire population and uses it to calculate the exact probability that any specific customer is still "alive."

## Step 2: Probability Modeling with BG/NBD

With the RFM matrix calibrated, we apply the **Beta Geometric / Negative Binomial Distribution (BG/NBD)** model. This framework models two distinct statistical processes simultaneously:
1. **The Purchasing Process:** While active, how frequently does a customer buy?
2. **The Dropout Process:** What is the probability that a customer permanently abandons the store after any given purchase?

By fitting this model to our matrix, we can calculate $P(Alive)$ (the probability that a customer hasn't churned) and predict exactly how many trips they will make in the next 30 days.

In [8]:
from lifetimes import BetaGeoFitter

# 1. Initialize the BG/NBD model
# We use a small penalizer coefficient to prevent overfitting on outlier customers
bgf = BetaGeoFitter(penalizer_coef=0.01)

# 2. Fit the model to our Recency, Frequency, and Age data
print("Training BG/NBD Model...")
bgf.fit(rfm_matrix['frequency'], rfm_matrix['recency'], rfm_matrix['T'])
print("✅ Model convergence successful!\n")

# 3. Calculate P(Alive) for every customer
rfm_matrix['P_Alive'] = bgf.conditional_probability_alive(
    rfm_matrix['frequency'],
    rfm_matrix['recency'],
    rfm_matrix['T']
)

# 4. Predict the exact number of purchases each customer will make in the next 30 days
t_days = 30
rfm_matrix['Predicted_Purchases_30_Days'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    t_days,
    rfm_matrix['frequency'],
    rfm_matrix['recency'],
    rfm_matrix['T']
)

# 5. Let's look at our highest "Flight Risk" customers
# We want customers who used to shop a lot (high frequency), but haven't been seen recently
flight_risks = rfm_matrix[(rfm_matrix['frequency'] > 10) & (rfm_matrix['P_Alive'] < 0.5)]
flight_risks = flight_risks.sort_values(by='P_Alive', ascending=True)

print(f"🚨 Found {len(flight_risks)} historically loyal customers who are likely lost.")
display(flight_risks[['frequency', 'recency', 'T', 'P_Alive', 'Predicted_Purchases_30_Days']].head(10))

Training BG/NBD Model...
✅ Model convergence successful!

🚨 Found 42 historically loyal customers who are likely lost.


/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,frequency,recency,T,P_Alive,Predicted_Purchases_30_Days
household_id,,,,,
2443,55.0,168.0,364.0,5.553528e-15,2.514354e-14
1974,78.0,225.0,365.0,9.426206e-13,5.994128e-12
2228,64.0,209.0,358.0,1.841766e-11,9.828802e-11
1031,74.0,174.0,267.0,4.624901e-10,3.783348e-09
370,67.0,174.0,268.0,5.789653e-09,4.280570e-08
1448,60.0,225.0,343.0,1.371906e-07,7.165750e-07
682,74.0,216.0,297.0,1.109315e-06,8.185720e-06
2212,28.0,164.0,350.0,2.394682e-06,5.865338e-06
2051,23.0,177.0,365.0,1.618311e-04,3.156303e-04


# Explanation:
Take Household 2443 at the very top of our list:They made 55 repeat purchases ($Frequency$).We have known them for 364 days ($T$).However, their last purchase was on day 168 ($Recency$).This means someone who used to shop constantly hasn't been seen in nearly 200 days.

The BG/NBD model mathematically confirms they are gone—giving them a $P(Alive)$ of essentially $0.00\%$. A standard churn model looking at a "30-day absence" would flag everyone, but this algorithm uniquely isolated the loyal customers who vanished.

## Step 3: Persona-Based Churn Intervention

Identifying silent attrition is only valuable if the business can act on it. By merging our $P(Alive)$ probabilities with our previously engineered Customer Personas, we can calculate the exact churn severity for each specific segment.

This allows marketing and operations teams to deploy highly targeted retention campaigns (e.g., sending high-value personalized coupons to Premium Shoppers whose $P(Alive)$ drops below 50%) rather than wasting budget on blanket discounts.

In [10]:
# 1. Load our master cluster labels
clusters = pd.read_csv('master_customers_fully_clustered.csv')

# 2. Merge the RFM probability matrix with the cluster labels
rfm_with_personas = rfm_matrix.reset_index().merge(
    clusters[['household_id', 'Hierarchical_Cluster']],
    on='household_id',
    how='inner'
)

# 3. Define an "At-Risk Loyalist"
# (Someone who shopped at least 5 times, but now has less than a 50% chance of being active)
rfm_with_personas['Is_At_Risk_Loyalist'] = (rfm_with_personas['frequency'] >= 5) & (rfm_with_personas['P_Alive'] < 0.5)

# 4. Aggregate the churn data by Persona
churn_summary = rfm_with_personas.groupby('Hierarchical_Cluster').agg(
    Total_Customers=('household_id', 'count'),
    Average_P_Alive=('P_Alive', 'mean'),
    At_Risk_Loyalists=('Is_At_Risk_Loyalist', 'sum')
).reset_index()

# 5. Calculate the specific Attrition Rate for these loyalists per segment
churn_summary['Loyalist_Attrition_Rate'] = (churn_summary['At_Risk_Loyalists'] / churn_summary['Total_Customers']) * 100

# Clean up names for presentation
churn_summary['Persona'] = churn_summary['Hierarchical_Cluster'].replace({
    0: 'Broad Middle',
    1: 'Power Shoppers',
    2: 'Premium Shoppers'
})

print("🚨 Silent Attrition Report by Persona:\n")
display(churn_summary[['Persona', 'Total_Customers', 'Average_P_Alive', 'At_Risk_Loyalists', 'Loyalist_Attrition_Rate']])

🚨 Silent Attrition Report by Persona:



,Persona,Total_Customers,Average_P_Alive,At_Risk_Loyalists,Loyalist_Attrition_Rate
0,Broad Middle,609,0.986121,7,1.149425
1,Power Shoppers,84,0.999378,0,0.000000
2,Premium Shoppers,108,0.999232,0,0.000000


#The Business Insight Breakdown
**The Power Shoppers (0% Attrition):** Once a customer in this segment reaches "loyalist" status (5+ purchases), they simply do not leave. Their routine of buying bulk staples is so deeply ingrained that they are essentially churn-proof.

**The Premium Shoppers (0% Attrition):** While these shoppers might visit less frequently, those who return 5+ times are highly dedicated to this specific store's specialty items. You do not easily abandon the store that sells your favorite imported cheese or premium wine.

**The Broad Middle (1.15% Attrition):** This is our battleground. These shoppers are likely the most price-sensitive and least loyal to your specific brand. If a competitor opens down the street with a coupon for 10% off groceries, the Broad Middle is the segment that quietly defects.